In [ ]:
!pip install -q prophet statsmodels pyarrow

In [ ]:
import itertools
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX


def find_input(filename, root="/kaggle/input"):
    matches = sorted(Path(root).rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one {filename} under {root}, found {len(matches)}: {matches}"
        )
    return matches[0]


features_df = pd.read_parquet(find_input("demandlens_features.parquet"))
features_df["date"] = pd.to_datetime(features_df["date"])
with open(find_input("subset_config.json"), encoding="utf-8") as f:
    config = json.load(f)

selected_stores = config["selected_stores"]
selected_families = config["selected_families"]
representative_combos = [tuple(x) for x in config["representative_combos"]]

In [ ]:
HORIZON = 15
N_FOLDS = 4


def get_walkforward_folds(all_dates, horizon=HORIZON, n_folds=N_FOLDS):
    all_dates = pd.DatetimeIndex(all_dates).sort_values()
    last_date = all_dates[-1]
    holdout_start = last_date - pd.Timedelta(days=horizon - 1)

    folds = []
    for i in range(n_folds, 0, -1):
        test_end = holdout_start - pd.Timedelta(days=(i - 1) * horizon + 1)
        test_start = test_end - pd.Timedelta(days=horizon - 1)
        train_end = test_start - pd.Timedelta(days=1)
        folds.append((train_end, test_start, test_end))

    holdout = (holdout_start - pd.Timedelta(days=1), holdout_start, last_date)
    return folds, holdout


all_dates = features_df["date"].unique()
folds, holdout = get_walkforward_folds(all_dates)
print("CV folds (train_end, test_start, test_end):")
for f in folds:
    print(f)
print("Holdout:", holdout)

In [ ]:
import numpy as np


def mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    mask = actual != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((actual[mask] - forecast[mask]) / actual[mask])) * 100


def wape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - forecast)) / denom * 100


def mase(actual, forecast, train_series, seasonal_period=7):
    actual, forecast = np.array(actual), np.array(forecast)
    train_series = np.array(train_series)
    naive_errors = np.abs(train_series[seasonal_period:] - train_series[:-seasonal_period])
    if len(naive_errors) == 0:
        return np.nan
    scale = np.mean(naive_errors)
    if scale == 0:
        return np.nan
    return np.mean(np.abs(actual - forecast)) / scale

In [ ]:
REGRESSORS = ["onpromotion", "is_holiday"]
MIN_TRAIN_ROWS = 60

prophet_results = []

for store_nbr in selected_stores:
    for family in selected_families:
        series_df = features_df[
            (features_df["store_nbr"] == store_nbr) & (features_df["family"] == family)
        ][["date", "sales"] + REGRESSORS].sort_values("date")
        series_df = series_df.rename(columns={"date": "ds", "sales": "y"})

        for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
            fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"

            train_df = series_df[series_df["ds"] <= train_end]
            test_df = series_df[(series_df["ds"] >= test_start) & (series_df["ds"] <= test_end)]
            if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
                continue

            m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
            for regressor in REGRESSORS:
                m.add_regressor(regressor)
            m.fit(train_df[["ds", "y"] + REGRESSORS])

            forecast = m.predict(test_df[["ds"] + REGRESSORS].copy())

            actual = test_df["y"].values
            pred = forecast["yhat"].clip(lower=0).values

            prophet_results.append({
                "store_nbr": store_nbr, "family": family, "fold": fold_label,
                "mape": mape(actual, pred), "wape": wape(actual, pred),
                "mase": mase(actual, pred, train_df["y"].values),
            })

prophet_results_df = pd.DataFrame(prophet_results)
print(prophet_results_df.groupby("fold")[["mape", "wape", "mase"]].mean())

In [ ]:
SARIMA_D = 0
SARIMA_SEASONAL_D = 1
SEASONAL_PERIOD = 7


def grid_search_sarima(train_series, train_exog, seasonal_period=SEASONAL_PERIOD):
    best_aic = np.inf
    best_order = None
    best_seasonal_order = None

    for p, q in itertools.product(range(2), range(2)):
        for P, Q in itertools.product(range(2), range(2)):
            order = (p, SARIMA_D, q)
            seasonal_order = (P, SARIMA_SEASONAL_D, Q, seasonal_period)
            try:
                fit = SARIMAX(
                    train_series, exog=train_exog, order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False,
                ).fit(disp=False)
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = order
                    best_seasonal_order = seasonal_order
            except Exception:
                continue
    return best_order, best_seasonal_order, best_aic


sarima_results = []

for store_nbr, family in representative_combos:
    frame = (
        features_df[(features_df["store_nbr"] == store_nbr) & (features_df["family"] == family)]
        .sort_values("date")
        .set_index("date")[["sales"] + REGRESSORS]
        .astype(float)
        .asfreq("D")
        .fillna(0)
    )
    series_df = frame["sales"]
    exog_df = frame[REGRESSORS]

    initial_mask = series_df.index <= folds[0][0]
    best_order, best_seasonal_order, best_aic = grid_search_sarima(
        series_df[initial_mask], exog_df[initial_mask]
    )
    print(f"Store {store_nbr}, {family}: best order={best_order}, seasonal={best_seasonal_order}, AIC={best_aic:.1f}")
    if best_order is None:
        print("No SARIMA candidate converged, skipping series")
        continue

    for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
        fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"
        train_mask = series_df.index <= train_end
        test_mask = (series_df.index >= test_start) & (series_df.index <= test_end)
        train_series, test_series = series_df[train_mask], series_df[test_mask]
        if len(train_series) < MIN_TRAIN_ROWS or len(test_series) == 0:
            continue

        try:
            fit = SARIMAX(
                train_series, exog=exog_df[train_mask], order=best_order,
                seasonal_order=best_seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False,
            ).fit(disp=False)
            pred = fit.forecast(steps=len(test_series), exog=exog_df[test_mask])
            pred = np.clip(pred.values, 0, None)
            actual = test_series.values

            sarima_results.append({
                "store_nbr": store_nbr, "family": family, "fold": fold_label,
                "order": str(best_order), "seasonal_order": str(best_seasonal_order),
                "mape": mape(actual, pred), "wape": wape(actual, pred),
                "mase": mase(actual, pred, train_series.values),
            })
        except Exception as e:
            print(f"SARIMA fit failed for {store_nbr}/{family}/{fold_label}: {e}")

sarima_results_df = pd.DataFrame(sarima_results)
print(sarima_results_df)

In [ ]:
sarima_results_df.to_csv("/kaggle/working/sarima_results.csv", index=False)
prophet_results_df.to_csv("/kaggle/working/prophet_results.csv", index=False)
print("Saved: prophet_results.csv, sarima_results.csv")